# Fit MIMIC image embeddings

Load a serialized vision dataset, fit MIMIC on the flattened image rows, compute embeddings with `transform`, and save those embeddings for later visualization.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "mimic_vision").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mimic import MIMIC
from mimic_vision import (
    VisionEmbedding,
    load_serialized_vision_dataset,
    resolve_artifact_file,
    save_vision_embedding,
)

In [3]:
DATASET_FILE = "last"  # Use "last" or an explicit dataset filename.
VISION_DATA_DIR = PROJECT_ROOT / "data" / "vision"
DATASET_DIR = VISION_DATA_DIR / "serialized"
EMBEDDING_DIR = VISION_DATA_DIR / "embeddings"
MODEL_DIR = VISION_DATA_DIR / "models"

MODE = "joint"  # options: "identity", "direct", "factorised", "joint"
CAPACITY = 0.25
RANDOM_STATE = 0
BOOTSTRAP = False
FEATURE_N_JOBS = -1

In [4]:
dataset_path = resolve_artifact_file(DATASET_FILE, input_dir=DATASET_DIR, pattern="*.pkl")
DATASET_FILE = dataset_path.name
dataset = load_serialized_vision_dataset(dataset_path)
DATASET_FILE, dataset.X.shape, dataset.images.shape, dataset.y.value_counts().sort_index()

('mnist_train_n4000_classes-5-6-8-9_21x21.pkl',
 (4000, 441),
 (4000, 21, 21),
 target
 5    1000
 6    1000
 8    1000
 9    1000
 Name: count, dtype: int64)

In [5]:
%%time

columns = {
    "regression": list(dataset.X.columns),
    "classification": [],
    "ignore": [],
}

model = MIMIC(
    columns=columns,
    mode=MODE,
    capacity=CAPACITY,
    bootstrap=BOOTSTRAP,
    feature_n_jobs=FEATURE_N_JOBS,
    random_state=RANDOM_STATE,
)
model.fit(dataset.X)

CPU times: user 14h 58min 39s, sys: 1h 15min 40s, total: 16h 14min 19s
Wall time: 4h 21min 27s


KeyboardInterrupt: 

In [ ]:
embeddings = model.transform(dataset.X)
embeddings.shape

In [ ]:
embedding_artifact = VisionEmbedding(
    dataset_file=DATASET_FILE,
    embeddings=embeddings,
    mode=MODE,
    capacity=CAPACITY,
    random_state=RANDOM_STATE,
)
saved_embedding_path = save_vision_embedding(embedding_artifact, output_dir=EMBEDDING_DIR)
saved_embedding_path.name

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_filename = saved_embedding_path.with_suffix(".joblib").name
saved_model_path = MODEL_DIR / model_filename
model.save(saved_model_path)
saved_model_path.name